In [ ]:
using Pkg
Pkg.activate("..")

using GeometricPreconditioner
using OptimalControl
using Plots
using ForwardDiff
using DifferentialEquations
using MINPACK
using Statistics

In [ ]:
# To be deplaced to utils.jl

# Function to dynamically plot the shooting function
function plot_shooting(S::Function, S₁::Function, S₂::Function, coord::Symbol)
    if coord == :S
        labels = ["S", "S₁", "S₂", "p₀", "p⁰"]
        xlim_S1 = [-7,2]
        xlim_S2 = [-1,1]
        xlim_S = [0,-1.5]
        ylim_S = [-8, 2]
    elseif coord == :T
        labels = ["T", "T₁", "T₂", "q₀", "q⁰"]
        xlim_S1 = [-3,3]
        xlim_S2 = [-1,1]
        xlim_S = [0,-1.5]
        ylim_S = [-3, 3]
    else
        error("Coordomates must be the old one (:S) or the new one (:T).")
    end
    η(p0) = -sqrt.(1 - p0.^2)                                       # Function η(⋅)
    S_(p⁰, p) = S([p⁰, p])

    # Plot S₁
    plt_S1 = plot(xlabel = labels[5], ylabel = labels[2], xlim = xlim_S1, legend=nothing)
    plot!(plt_S1, S₁, c = 2, lw = 2)

    # Plot S₂
    plt_S2 = plot(xlabel = labels[4], ylabel = labels[3], xlim = [-1,1], legend=nothing)
    plot!(plt_S2, S₂, c = 3, lw = 2)

    # Plot S
    plt_S = surface(xlabel = labels[5], ylabel = labels[4], zlabel = labels[1], xflip = true)
    surface!(plt_S, range(xlim_S[1], xlim_S[2], 100), range(ylim_S[1], ylim_S[2], 100), S_, camera = (30,30))
    plot3d!(plt_S, -1*ones(100), range(ylim_S[1], ylim_S[2], 100), S₁.(range(ylim_S[1], ylim_S[2], 100)), label = labels[2], lw = 2)
    plot3d!(plt_S, η.(range(-1, 1, 100)), range(-1, 1, 100), S₂.(range(-1, 1, 100)), label = labels[3], lw = 2)

    plt_S12 = plot(plt_S1, plt_S2, layout = (1,2))
    plt_total = plot(plt_S, plt_S12, layout = grid(2,1, heights = [2/3, 1/3]), size=(800, 600))

    return plt_total
end

function fit_ellipse(x, y)
    M = hcat(x.^2, x.*y, y.^2, x, y)            # quadratic form of ellipse
    p = M\ones(length(x))                       # fit parameters for the ellipse
    A, B, C, D, E = p
    F = -1.0
    # calculate the parameters from quadratic:
    Δ = B^2 - 4*A*C
    Λ = (A-C)^2 + B^2
    b, a = [-sqrt(clamp( 2*(A*E^2 + C*D^2 - B*D*E + Δ*F)*
            ( (A+C) + op(sqrt(Λ)) ), 0, Inf)) / Δ   for op in (+, -)]
    θ = atan(-B, C-A)/2
    c = [(2*C*D - B*E)/Δ, (2*A*E - B*D)/Δ]
    return a, b, -θ+Base.π/2, c
end

function plot_sol(sol, size = (800, 600))

    # plot
    t = sol.t
    x⁰ = [sol.u[i][1] for i in 1:length(sol.u)]
    x  = [sol.u[i][2] for i in 1:length(sol.u)]
    p⁰ = [sol.u[i][3] for i in 1:length(sol.u)]
    p  = [sol.u[i][4] for i in 1:length(sol.u)]
    u = sign.(p)

    plt_x⁰ = plot(t, x⁰, ylabel = "x⁰", label = nothing, lw = 2, title = "state", titlefontsize = 10)
    plt_x  = plot(t, x , ylabel = "x", label = nothing, lw = 2)
    plt_p⁰ = plot(t, p⁰, ylim=[-1,1], lw = 2, label = nothing, title = "costate", titlefontsize = 10)
    plt_p  = plot(t, p, label = nothing, lw = 2)
    plt_u  = plot(t, u , xlabel = "u", title = "control", titlefontsize = 10, label = nothing, lw = 2)

    plt_xp = plot(plt_x⁰, plt_p⁰, plt_x, plt_p, layout=(2, 2))
    return plot(plt_xp, plt_u, layout = grid(2,1, heights = [2/3, 1/3]), size=size)
end

In [ ]:
global α = 1

# Event when condition(z,t,integrator) == 0
function condition(z, t, integrator)                        
    x⁰,x,p⁰,p = z
    return p
end

# Action when condition(z,t,integrator) == 0 
function affect!(integrator)                                
    global α = -α
    nothing
end

cb = ContinuousCallback(condition, affect!)                     # Callback
H(x, p) = p[1] * x[2] + α * p[2]                                # Hamiltonian
ϕ_ = Flow(OptimalControl.Hamiltonian(H), callback = cb)         # Flow with maximizing control 

# Flow for shooting function
function ϕ(t0, x0, p0, tf; kwargs...)                       
    if p0[2] == 0
        global α = -sign(p0[1])
    else
        global α = sign(p0[2])
    end
    return ϕ_(t0, x0, p0, tf; kwargs...)
end

# Flow for plot
function ϕ((t0, tf), x0, p0; kwargs...)                     
    if p0[2] == 0
        global α = -sign(p0[1])
    else
        global α = sign(p0[2])
    end
    return ϕ_((t0, tf), x0, p0; kwargs...)
end

In [ ]:
t0 = 0                                                                      # Initial time
x0 = [0,0]                                                                  # Initial augmented state
tf = 5                                                                      # Final time
global xT = 0                                                               # Final state

π((x,p)) = x[2]                                                             # projection on state space
η(p0) = -sqrt.(1 - p0.^2)                                                   # Function η(⋅)

S(p0; xT=xT) = π( ϕ(t0, x0, p0, tf) ) - xT                                  # General shooting function

S₁(p0; xT=xT) = S([-1, p0]; xT=xT)                                          # Normalization 1
S₂(p0; xT=xT) = abs(p0) < 1 ? S([η(p0), p0]; xT=xT) : sign(p0)*tf - xT      # Normalization 2

# Plot shooting functions
plt = plot_shooting(S, S₁, S₂, :S)
@gif for i ∈ [range(30, 90, 50); 90*ones(25); range(90, 30, 50); 30*ones(25)]
    plot!(plt[1], camera=(i,i), 
        zticks = i==90 ? false : true,
        zlabel = i==90 ? "" : "S" )
end

As done before, we use the solver $\texttt{hybrd1}$ from the $\texttt{MINPACK.jl}$ package to find a zero of $S_2$.

In [ ]:
global iterate_S2 = Vector{Float64}()                       # global vector to store iterates of the solver

function S₂!(s₂, ξ; xT=xT)                                         # intermediate function
    push!(iterate_S2, ξ[1])                                 # stock the iterate
    return (s₂[:] .= S₂(ξ[1]; xT=xT); nothing)                     
end

JS₂(ξ; xT=xT) = ForwardDiff.jacobian(p0 -> [S₂(p0[1]; xT=xT)], ξ)         # compute jacobian by forward differentiation
JS₂!(js₂, ξ; xT=xT) = (js₂[:] .= JS₂(ξ; xT=xT); nothing)                  # intermediate function

ξ = [-0.5]                                                  # initial guess
p0_sol = fsolve(S₂!, JS₂!, ξ, show_trace = true)            # solve

In [ ]:
# Get the optimal trajectory
sol = ϕ((t0, tf), x0, [η(p0_sol.x[1]), p0_sol.x[1]], saveat=range(t0, tf, 500))
# Plot the solution
plot_sol(sol)

In [ ]:
function plot_ellipse(a,b,θ,c,φ,x)
    # Data for plot 
    # Boundary of the accessible set
    n_ = 100                                            # Number of points
    p0_ = [[[-1, i] for i ∈ range(-tf, 0, n_)];         # Initial costate
        [[1, i] for i ∈ range(tf, 0, n_)]]
    x_ = zeros(2, 2*n_); p_ = zeros(2, 2*n_)            # Init final state and costate 
    for i = 1:length(p0_)
        x_[:,i], p_[:,i] = ϕ(t0, x0, p0_[i], tf)        # Compute flow for plot
    end

    # Ellipse
    β = range(-Base.π, Base.π; length = 100)            # Angle for plot the ellipse
    xₑ = r(-θ)*s(a,b)*
        transpose(reduce(hcat,[sin.(β), cos.(β)])).+c   # points of the ellipse

    # Transport x, x_ and xₑ on the new coordinates
    y = φ(x); y_ = φ(x_); yₑ = φ(xₑ)                    

    # plot
    plt_x = plot(x_[2,:], x_[1,:], label = nothing)
    scatter!(x[2,:], x[1,:], label="Observations", legend = :topleft)
    plot!(xₑ[2,:], xₑ[1,:], label = "Fitted ellipse")
    plot!(xlim = [-15,15], ylim = [-15,15], xlabel = "x", ylabel = "x⁰")

    plt_y = plot(y_[2,:], y_[1,:], label = nothing)
    scatter!(y[2,:], y[1,:], label="")
    plot!(yₑ[2,:], yₑ[1,:], label = "")
    plot!( xlabel = "y", ylabel = "y⁰")

    return plot(plt_x, plt_y, layout = (1,2), size=(800, 400))
end

In [ ]:
# For fitting data
n = 15                                              # Number of points
p0 = [[[-1, i] for i ∈ range(-tf, 0, n)];           # Initial costate
      [[1, i] for i ∈ range(tf, 0, n)]]    
x = zeros(2, 2*n); p = zeros(2,2*n)                 # Init final state and costate 
for i = 1:length(p0)
    x[:,i], p[:,i] = ϕ(t0, x0, p0[i], tf)           # Compute flow
end

a, b, θ, c = fit_ellipse(x[1,:], x[2,:])            # Fit ellipse
r(β) = [[cos(β), sin(β)] [-sin(β), cos(β)]]         # 2x2 rotation matrix
s(a,b) = [[a,0] [0,b]]                              # 2x2 scale matrix

# Construction of the linear diffeomorphism ϕ(x) = Ax * B
d = (a*sin(θ))/(b*cos(θ)); β₀ = atan(d)             # intermediate values
A = r(-β₀)*s(1/a,1/b)*r(θ); B = -A*c                # calculate A and B
φ(x) = A*x .+ B                                     # diffeomorphism

plot_ellipse(a, b, θ, c, φ, x)

In [ ]:
p₀(p) = [p[1], p[2] + p[1]*tf]                                              # Function p₀(⋅)
Aₓ = A[2,2]; Bₓ = B[2]                                                      # Aₓ and Bₓ
yT = Aₓ*xT + Bₓ                                                             # Target in the new coordinate 
T(q; xT=xT) = Aₓ*(S(p₀(transpose(A)*q); xT=xT))                             # State flow in the new coordinates
T₁(q; xT=xT) = T([-1, q]; xT=xT)                                            # Normalization 1
T₂(q; xT=xT) = abs(q) < 1 ? T([η(q), q]; xT=xT) : sign(q)*(Aₓ*tf + Bₓ)      # Normalization 2

# Plot shooting functions
plt = plot_shooting(T, T₁, T₂, :T)
# @gif for i ∈ [range(30, 90, 50); 90*ones(25); range(90, 30, 50); 30*ones(25)]
#     plot!(plt[1], camera=(i,i), 
#         zticks = i==90 ? false : true,
#         zlabel = i==90 ? "" : "T" )
# end

In [ ]:
global iterate_T2 = Vector{Float64}()                       # global vector to store iterates of the solver

function T₂!(t₂, ξ; xT=xT)                                         # intermediate function
     push!(iterate_T2, ξ[1])  
     return (t₂[:] .= T₂(ξ[1]; xT=xT); nothing)                   
end

JT₂(ξ; xT=xT) = ForwardDiff.jacobian(q0 -> [T₂(q0[1]; xT=xT)], ξ)         # compute jacobian by forward differentiation
JT₂!(jt₂, ξ; xT=xT) = (jt₂[:] .= JT₂(ξ; xT=xT); nothing)                  # intermediate function

ξ = [0.5]                                                   # initial guess
q_sol = fsolve(T₂!, JT₂!, ξ, show_trace = true)             # solve

In [ ]:
p0_sol = p₀(transpose(A)*[η(q_sol.x[1]), q_sol.x[1]])       # get the optimal initial costate in old coordinates
sol = ϕ((t0, tf), x0, p0_sol, saveat=range(t0, tf, 500))    # get optimal trajectory

plot_sol(sol)

It is shown in [mettre article] that if the boundary of augmented accessible set is the fitted ellipse then the shooting function $T_2$ is the identity function. Due to the error of the approximation, the function $T_2$ is not the identity, but we hope that this function is close to this ideal function, and so that the convergence of $T_2$ is faster than the one of $S_2$. 

The following code compares the convergence of these two shooting function. We also study for these functions the domain of initial guesses for the solver that hit the bounds $[-1, 1]$ during the solving process. 

In [ ]:
N = 100
xT_span = range(minimum(x[2,:]), maximum(x[2,:]), length=N)
# initialization of the matrix
fnorms_S2 = zeros(N, 100) 
fnorms_T2 = zeros(N, 100)        
fnorms_T2_IG = zeros(N, 100)

# Intermediate function to get value of S from T iterates
T₂_(q0) = abs(q0) < 1 ? S(p₀(transpose(A)*[η.(q0),q0])) : sign(q0) * tf - xT

# Initial guess
ξ = 0.5
    
global xT = 3; #xT_span[i]

p0_sol_S2 = fsolve(S₂!, JS₂!, [ξ], show_trace = true, tracing = true)
# p0_sol_S2 = fsolve(
#     (s₂, ξ) -> s₂ .= S₂!(s₂, ξ; xT=xT),
#     (js₂, ξ) -> js₂ = JS₂!(js₂, ξ; xT=xT),
#     [ξ], show_trace=false, tracing=true)
println(p0_sol_S2)

plot(S₂, -1, 1)
#plot!(x ->JS₂([x])[1])



In [ ]:
## initial guesses
N = 1000; ξ = range(-1.,1.,N)                               

# initialization of the matrix
fnorms_S2 = zeros(N, 100); fnorms_T2 = zeros(N, 100)        
fnorms_T2_in_x = zeros(N, 100)
iterates_S2 = zeros(N, 100); iterates_T2 = zeros(N, 100)
conv_S2 = zeros(N,1); conv_T2 = zeros(N,1)                  
#-1 : not converged; 1 converged; 0: converged but hit bounds

# intermediate function to get value of S from T iterates
T₂_(q0) = abs(q0) < 1 ? S(p₀(transpose(A)*[η.(q0),q0])) : sign(q0) * tf - xT

for i = 1:N
    # remove old iterates
    global iterate_S2 = Vector{Float64}()
    global iterate_T2 = Vector{Float64}()

    # solve 
    q_sol_S2 = fsolve(S₂!, JS₂!, [ξ[i]], show_trace = false, tracing = true)
    q_sol_T2 = fsolve(T₂!, JT₂!, [ξ[i]], show_trace = false, tracing = true)
    

    # store results is converged 
    if q_sol_S2.converged
        fnorm_S2 = [q_sol_S2.trace.trace[j].fnorm for j ∈ 1:length(q_sol_S2.trace.trace)]
        iterates_S2[i,1:length(iterate_S2)] = iterate_S2
        conv_S2[i] = length(findall(x-> abs(x) > 1, iterate_S2)) == 0
        fnorms_S2[i,1:length(fnorm_S2)] = fnorm_S2
    else
        conv_S2[i] = -1
    end
    if q_sol_T2.converged
        fnorm_T2 = [q_sol_T2.trace.trace[j].fnorm for j ∈ 1:length(q_sol_T2.trace.trace)]
        iterates_T2[i,1:length(iterate_T2)] = iterate_T2
        conv_T2[i] = length(findall(x-> abs(x) > 1, iterate_T2)) == 0
        fnorms_T2[i,1:length(fnorm_T2)] = fnorm_T2 
        fnorms_T2_in_x[i, 1:length(iterate_T2)] = abs.(T₂_.(iterate_T2))
    else
        conv_T2[i] = -1
    end
end

# mean 
mean_fnorms_S2 = mean(fnorms_S2, dims = 1)
mean_fnorms_T2 = mean(fnorms_T2, dims = 1)
mean_fnorms_T2_in_x = mean(fnorms_T2_in_x, dims = 1)

# remove zeros with tolerance ε 
ε = 1e-9;
mean_fnorms_S2 = mean_fnorms_S2[1:findall(x -> x < ε, mean_fnorms_S2)[1][2]]
mean_fnorms_T2 = mean_fnorms_T2[1:findall(x -> x < ε, mean_fnorms_T2)[1][2]]
mean_fnorms_T2_in_x = mean_fnorms_T2_in_x[1:findall(x -> x < ε, mean_fnorms_T2_in_x)[1][2]]

# plots
plt_1 = plot(0:length(mean_fnorms_S2)-1, mean_fnorms_S2, label = "S₂")
plot!(0:length(mean_fnorms_T2)-1, mean_fnorms_T2, label = "T₂")
plot!(0:length(mean_fnorms_T2_in_x)-1, mean_fnorms_T2_in_x, label = "T₂ in x")
plot!(yaxis = :log10, xlim = [0, 30], ylim = [ε, 10], xlabel = "Error", ylabel = "Iterations")

plt_21 = plot(ξ, S₂, color = :black, label = "")
color = [conv_S2[i]==1 ? :green : conv_S2[i] == 0 ? :blue : :red for i ∈ 1:N]
scatter!(ξ, S₂, color = color, markerstrokecolor = color, marker = 2, label ="")
plot!([-1,1], [xT,xT], color = :black, label = "")
plot!(xlabel = "p₀", ylabel = "S₂", label = "")

plt_22 = plot(ξ, T₂, color = :black, label = "")
color = [conv_T2[i]==1 ? :green : conv_T2[i] == 0 ? :blue : :red for i ∈ 1:N]
scatter!(ξ, T₂, markercolor = color, markerstrokecolor = color, marker = 2, label = "")
plot!([-1,1], [yT,yT], color = :black, label = "")
plot!(xlabel = "q₀", ylabel = "T₂", label = "")
scatter!(1,1, markercolor = :green, markerstrokecolor = :green, marker = 2, label = "converged")
scatter!(1,1, markercolor = :blue, markerstrokecolor = :blue, marker = 2, label = "converged but hit bounds")
scatter!(1,1, markercolor = :red, markerstrokecolor = :red, marker = 2, label = "not converged")

plt_2 = plot(plt_21, plt_22, layout = (1,2))
plot(plt_1, plt_2, layout = grid(2,1, heights = [0.5, 0.5]), size=(700, 600))